In [14]:
%%writefile vector_sum.cu
#include <iostream>
#include <vector>
#include <chrono>
#include <cuda_runtime.h>
#include <iomanip>

using namespace std;

// ================= CPU =================
float sum_cpu(const vector<float>& v) {
    float sum = 0;
    for (float x : v) sum += x;
    return sum;
}

// ================= GPU =================
__global__ void sum_gpu(float* input, float* result, int N) {
    __shared__ float cache[256];

    int tid = threadIdx.x + blockIdx.x * blockDim.x;
    int cacheIdx = threadIdx.x;

    float temp = 0;
    while (tid < N) {
        temp += input[tid];
        tid += blockDim.x * gridDim.x;
    }

    cache[cacheIdx] = temp;
    __syncthreads();

    int i = blockDim.x / 2;
    while (i != 0) {
        if (cacheIdx < i)
            cache[cacheIdx] += cache[cacheIdx + i];
        __syncthreads();
        i /= 2;
    }

    if (cacheIdx == 0)
        atomicAdd(result, cache[0]);
}


int main() {

    cout << setw(12) << "N"
         << setw(15) << "CPU(s)"
         << setw(15) << "GPU(ms)"
         << setw(15) << "Speedup"
         << setw(20) << "CPU sum"
         << setw(20) << "GPU sum"
         << endl;

    for (int N = 1000; N <= 1000000; N += 1000) {

        cout << "\n=== Iteration N = " << N << " ===" << endl;

        vector<float> v(N, 1.0f);

        // ===== CPU =====
        auto cpu_start = chrono::high_resolution_clock::now();
        float cpu_sum = sum_cpu(v);
        auto cpu_end = chrono::high_resolution_clock::now();

        double cpu_time = chrono::duration<double>(cpu_end - cpu_start).count();

        // ===== GPU =====
        float *d_v, *d_result;
        cudaMalloc(&d_v, N * sizeof(float));
        cudaMalloc(&d_result, sizeof(float));

        cudaMemcpy(d_v, v.data(), N * sizeof(float), cudaMemcpyHostToDevice);
        cudaMemset(d_result, 0, sizeof(float));

        cudaEvent_t start, stop;
        cudaEventCreate(&start);
        cudaEventCreate(&stop);

        cudaEventRecord(start);

        sum_gpu<<<256, 256>>>(d_v, d_result, N);

        cudaEventRecord(stop);
        cudaEventSynchronize(stop);

        float gpu_time;
        cudaEventElapsedTime(&gpu_time, start, stop);

        float gpu_sum;
        cudaMemcpy(&gpu_sum, d_result, sizeof(float), cudaMemcpyDeviceToHost);

        double speedup = cpu_time / (gpu_time / 1000.0);

        cout << setw(12) << N
             << setw(15) << cpu_time
             << setw(15) << gpu_time
             << setw(15) << speedup
             << setw(20) << cpu_sum
             << setw(20) << gpu_sum
             << endl;

        cudaFree(d_v);
        cudaFree(d_result);
    }

    return 0;
}

Overwriting vector_sum.cu


In [15]:
!nvcc vector_sum.cu -o vsum
!./vsum

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
           N         CPU(s)        GPU(ms)        Speedup             CPU sum             GPU sum

=== Iteration N = 1000 ===
        1000      1.469e-05       0.192896       0.076155                1000                1000

=== Iteration N = 2000 ===
        2000     2.9262e-05       0.016672        1.75516                2000                2000

=== Iteration N = 3000 ===
        3000     4.3696e-05        0.01488        2.93656                3000                3000

=== Iteration N = 4000 ===
        4000     5.6766e-05       0.016672        3.40487                4000                4000

=== Iteration N = 5000 ===
        5000     8.3438e-05       0.016448        5.07284                5000                5000

=== Iteration N = 6000 ===
        6000     8.6941e-05       0.022528        3.85924